In [1]:
### Kenyan Parliamentary Hansard Analysis Script

## Nicolas de Zamaroczy

## The purpose of this Python script is to perform simple quantitative text analysis of the parsed Kenyan parliamentary hansards.
# In particular, we track sentiment analysis in a variety of ways:
# - over time;
# - by speaker; and
# - by political party.

# The current goal is to see if parliamentary speech is a leading or lagging indicator of political violence in Kenya.

In [1]:
### PRELIMINARIES

# --- Dependency Checks ---
# This block checks if several required libraries are installed.
# If not, it uses Python's package manager (pip) to install them.
# This makes the script more portable and easier for others to run.
try:
    import numpy as np # NumPy, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("NumPy not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    import numpy as np
    print("Installation complete.")

try:
    import pandas as pd # Pandas, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("Pandas not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
    print("Installation complete.")

try:
    from pathlib import Path # PathLib, used to read/write files to a given folder
except ImportError:
    import subprocess
    import sys
    print("PathLib not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pathlib[Path]"])
    from pathlib import Path
    print("Installation complete.")

try:
    from tqdm import tqdm # tqdm, used to create a progress bar
except ImportError:
    import subprocess
    import sys
    print("tqdm not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm[tqdm]"])
    from tqdm import tqdm
    print("Installation complete.")

# Enable tqdm for Pandas
tqdm.pandas()

# Disable Future Deprecation Warnings from getting logged
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

In [3]:
# Load the csv with parsed speech blocks

# Current notebook directory
notebook_dir = Path.cwd()
# Path to your CSV
file_path = notebook_dir.parent / "outputs" / "senate_parser_outputs.csv"
#file_path = notebook_dir.parent / "outputs" / "national_assembly_parser_outputs.csv"
# Check existence and raise an error if missing
if not file_path.exists():
    raise FileNotFoundError(f"CSV file not found: {file_path}")

# Read the CSV
df_blocks = pd.read_csv(file_path)

# Remove any \n artifacts that are hidden embeds in the content field
df_blocks["content"] = df_blocks["content"].str.replace(r"\n", " ", regex=True).str.replace(r"\r", " ", regex=True)

# Drop some fields that aren't relevant to the analysis
df_blocks = df_blocks[['filename','date','speaker','matched_name','language_distribution','content','is_continuation']]

# Ensure date column is a datetime
# We're currently getting some messy data in the date field, need to check what's causing it upstream
df_blocks["date"] = pd.to_datetime(df_blocks["date"], errors="coerce")
df_blocks = df_blocks.dropna(subset=["date"])

df_blocks

,filename,date,speaker,matched_name,language_distribution,content,is_continuation
0,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","Hon. Senators, let us begin with the statement...",False
1,25th_November_2015.pdf,2015-11-25,Sen. Haji,NaN,"{""en"": 1.0}","Mr. Speaker, Sir, the statement is not ready. ...",False
2,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","What do you have to say, Sen. Mositet?",False
3,25th_November_2015.pdf,2015-11-25,Sen. Mositet,NaN,"{""sw"": 1.0}","Bw. Spika, jana Naibu wa Mwenyeketi alitueleza...",False
4,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","Order, Senator! The statement will be issued o...",False
...,...,...,...,...,...,...,...
209978,Wednesday__3rd_December_2014.pdf,2014-12-03,Sen. Ndiema,NaN,"{""en"": 1.0}","Thank you, Madam Temporary Speaker. I am seeki...",False
209979,Wednesday__3rd_December_2014.pdf,2014-12-03,The Temporary Speaker (Sen. Ongoro),NaN,"{""en"": 1.0}","Senator, how did you know and yet you do not h...",False
209980,Wednesday__3rd_December_2014.pdf,2014-12-03,Sen. Ndiema,NaN,"{""en"": 1.0}","Madam Temporary Speaker, I do not know whether...",False
209981,Wednesday__3rd_December_2014.pdf,2014-12-03,The Temporary Speaker (Sen. Ongoro),NaN,"{""en"": 1.0}","Unfortunately, you did not manage to conclude ...",False


In [5]:
# Join additional information about speakers from the Google spreadsheet

# There's some parsing error that happens when you trying reading the Google spreadhseet directly
#googlesheet_url = 'https://docs.google.com/spreadsheets/d/1zVHI2mlEpPHtwvCanyzfEjKuYJEQmHGrLo1jQNa4jrE/edit?usp=sharing'
#df_googlesheet = pd.read_csv(googlesheet_url, quoting=3, engine="python")
#df_googlesheet

# So instead we manually export the Google sheet as a TSV and read that in
file_path = notebook_dir.parent / "inputs" / "parliamentarians" / "googlesheet.tsv"
# Check existence and raise an error if missing
if not file_path.exists():
    raise FileNotFoundError(f"TSV file not found: {file_path}")
df_googlesheet = pd.read_csv(file_path, sep="\t")
df_googlesheet.columns = df_googlesheet.columns.str.lower()
#df_googlesheet

# Merge
# !!! Eventually we should add logic here to limit matching to only allow matches on date (i.e. if a given session matches to a speaker's dates in office)

df_googlesheet["speaker_name"] = (df_googlesheet["first name(s)"].str.strip() + " " + df_googlesheet["last name"].str.strip()).str.lower()
df_blocks["matched_name"] = df_blocks["matched_name"].str.strip().str.lower()
df_blocks = df_blocks.merge(df_googlesheet, left_on="matched_name", right_on="speaker_name", how="left")

df_blocks

,filename,date,speaker,matched_name,language_distribution,content,is_continuation,session,chamber,full name,...,county,constituency,party affiliation,date took office,date left office,leadership role,county woman representative,nominated members,note,speaker_name
0,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","Hon. Senators, let us begin with the statement...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25th_November_2015.pdf,2015-11-25,Sen. Haji,NaN,"{""en"": 1.0}","Mr. Speaker, Sir, the statement is not ready. ...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","What do you have to say, Sen. Mositet?",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25th_November_2015.pdf,2015-11-25,Sen. Mositet,NaN,"{""sw"": 1.0}","Bw. Spika, jana Naibu wa Mwenyeketi alitueleza...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25th_November_2015.pdf,2015-11-25,The Speaker (Hon. Ethuro),NaN,"{""en"": 1.0}","Order, Senator! The statement will be issued o...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188642,Wednesday__3rd_December_2014.pdf,2014-12-03,Sen. Ndiema,NaN,"{""en"": 1.0}","Thank you, Madam Temporary Speaker. I am seeki...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188643,Wednesday__3rd_December_2014.pdf,2014-12-03,The Temporary Speaker (Sen. Ongoro),NaN,"{""en"": 1.0}","Senator, how did you know and yet you do not h...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188644,Wednesday__3rd_December_2014.pdf,2014-12-03,Sen. Ndiema,NaN,"{""en"": 1.0}","Madam Temporary Speaker, I do not know whether...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188645,Wednesday__3rd_December_2014.pdf,2014-12-03,The Temporary Speaker (Sen. Ongoro),NaN,"{""en"": 1.0}","Unfortunately, you did not manage to conclude ...",False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Perform multilingual sentiment analysis using batching

# Here is a direct link to tabularisai's multilingual-sentiment-analysis package, hosted on HuggingFace:
# https://huggingface.co/tabularisai/multilingual-sentiment-analysis

# Tabularisai's MSA tool automatically detects speech, so in theory it doesn't need the information contained in language_distribution.

# Import pipeline
from transformers import pipeline

# Set up the pipeline
pipe = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis"
)

# Set up batching
texts = df_blocks["content"].astype(str).tolist()
batch_size = 32

# Run pipeline
all_results = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    #results = pipe(batch)
    results = pipe(batch, truncation=True, max_length=510) # This has a saftey valve built in to not hit the 512 max tokens limit Tabularisai has, but shouldn't be necessary if the previous script works as intended
    all_results.extend(results)

df_blocks["sentiment"] = [r["label"] for r in all_results]
df_blocks["confidence_score"] = [r["score"] for r in all_results]

df_blocks

KeyboardInterrupt: 

In [ ]:
# Visualizing the distribution of confidence scores

palette = {
    "Very Negative": "#d73027",
    "Negative": "#fc8d59",
    "Neutral": "#999999",
    "Positive": "#91cf60",
    "Very Positive": "#1a9850"
}

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.histplot(
    data=df_blocks,
    x="confidence_score",
    hue="sentiment",
    bins=30,
    kde=True,
    palette=palette
)
plt.title("Distribution of Confidence Scores by Class")
plt.xlabel("Confidence Score")
plt.ylabel("Count")
plt.show()

In [ ]:
# Checking to see if sentiment analysis passes the sniff test

# Define all sentiment classes
sentiment_classes = ["Very Negative", "Negative", "Neutral", "Positive", "Very Positive"]

# Function to safely sample up to 5 rows
def safe_sample(df, label, n=5):
    subset = df[df["sentiment"] == label]
    return subset.sample(n=min(len(subset), n), random_state=42)

# Change display settings to allow full text to show instead of truncating
pd.set_option('display.max_colwidth', None)

# Print sample content for each class
for label in sentiment_classes:
    print(f"\n--- {label} ---")
    sample = safe_sample(df_blocks, label, 5)
    display(sample[["content", "confidence_score"]])  # use display() in Jupyter


In [ ]:
# Construct a single measure of sentiment for over-time analysis

# Reset our display options to their defaults
pd.reset_option("display.max_rows")
pd.reset_option("display.max_colwidth")

scale_map = {
    "Very Negative": -1,
    "Negative": -0.5,
    "Neutral": 0,
    "Positive": 0.5,
    "Very Positive": 1
}

df_blocks["sentiment_scaled"] = df_blocks["sentiment"].map(scale_map) * df_blocks["confidence_score"]

# Exclude all the neutral sentiment blocks (which is the overwhelming majority, so this greatly pares down df_blocks)
df_blocks_w_sentiment = df_blocks[df_blocks['sentiment']!='Neutral'].copy()

df_blocks_w_sentiment

In [ ]:
# Group by week
df_blocks_w_sentiment["date"] = pd.to_datetime(df_blocks_w_sentiment["date"])

# Group by week (ending on Sunday by default) and average the scaled sentiment
weekly_sentiment = (
    df_blocks_w_sentiment
    .set_index("date")
    .resample("W")["sentiment_scaled"]
    .mean()
    .reset_index()
)

weekly_sentiment

In [ ]:
# Visualize

import plotly.express as px

fig = px.line(
    weekly_sentiment,
    x="date",
    y="sentiment_scaled",
    title="Weekly Sentiment Over Time",
    labels={"date": "Week", "sentiment_scaled": "Avg Sentiment (scaled)"}
)

fig.update_traces(mode="lines+markers")  # adds points to each week
fig.show()

In [ ]:
# ==============================================================================
# --- VALIDATION QUERIES ---
# Useful queries are stored here for subsequent use.
# ==============================================================================

# Find examples of code-switching (i.e. when a speaker switches languages mid-speech)
#df_codeswitching = all_speaker_data[all_speaker_data['is_continuation']==TRUE]

# Check that there isn't a language bias in our sentiment analysis

# Look to see which parties have the highest and lowest sentiment

#  Is it worth introducing confidence thresholds


In [17]:
# ==============================================================================
# --- ONE SHOT ENCODING ---
# ==============================================================================

############### This is the right approach for multilingual zero-shot classification, but takes way too long to run locally >24h).

# We use the ############### Hugging Face transformer to do zero-shot classification since it is multilingual and supports Swahili as a language.
# See ########### for documentation

# Set up the pipeline
from transformers import pipeline

classifier = pipeline("zero-shot-classification"
                      ,model="MoritzLaurer/multilingual-MiniLMv2-L12-mnli-xnli")

# Define topic labels for zero-shot classification
labels = [
    "parliamentary process",
    "economy and finance",
    "education",
    "healthcare",
    "foreign affairs",
    "security and defense",
    "governance and corruption",
    "environment and climate",
    "human rights and equality",
    "infrastructure projects",
    "agriculture and food security",
    "devolution and local government"
]

# Set up batching
batch_size = 8
results = []

# Execute
for i in tqdm(range(0, len(df_blocks), batch_size)):
    batch = df_blocks["content"].iloc[i:i+batch_size].tolist()
    batch_results = classifier(batch, labels)  # supports batch inputs

    for res in batch_results:
        results.append((res["labels"][0], res["scores"][0]))

df_blocks[["predicted_label", "confidence"]] = pd.DataFrame(results, index=df_blocks.index)

config.json: 0.00B [00:00, ?B/s]

C:\Users\USER\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--MoritzLaurer--multilingual-MiniLMv2-L12-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

  2%|█▋                                                                         | 532/23581 [23:38<17:04:08,  2.67s/it]


KeyboardInterrupt: 

In [7]:
## English-only speedy zero-shot classification

# English-only subset
df_en = df_blocks[df_blocks["language_distribution"] == {"en": 1.0}].copy()

# Truncate to keep only first 500 characters
df_en["content_trunc"] = df_en["content"].str.slice(0, 500)

# Import pipeline
from transformers import pipeline

# Load a fast, English-only zero-shot model
classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=-1
)

# # Debugging block
# test_text = "The government proposed new tax reforms to boost the economy."
# labels = [
#     "economy and finance",
#     "education",
#     "security and defense"
# ]
# result = classifier(test_text, labels)
# print(result)

# Define labels
labels = [
    "parliamentary process",
    "economy and finance",
    "education",
    "healthcare",
    "foreign affairs",
    "security and defense",
    "governance and corruption",
    "environment and climate",
    "human rights and equality",
    "infrastructure projects",
    "agriculture and food security",
    "devolution and local government"
]

# Batched zero-shot classification
batch_size = 8
predicted_labels = []
confidences = []

for i in tqdm(range(0, len(df_en), batch_size)):
    batch_texts = df_en["content_trunc"].iloc[i:i+batch_size].tolist()
    batch_results = classifier(batch_texts, candidate_labels=labels)
    
    # Ensure consistent output format
    if isinstance(batch_results, dict):  # Single input → single dict
        batch_results = [batch_results]
    
    for r in batch_results:
        predicted_labels.append(r["labels"][0])
        confidences.append(r["scores"][0])

# Assign back
df_en = df_en.assign(predicted_label=predicted_labels, confidence=confidences)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\USER\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.